In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
base_path = r"D:\project\G1020"
image_dir = os.path.join(base_path, "Images")
csv_path = os.path.join(base_path, "G1020.csv")

In [3]:
# Load labels
df = pd.read_csv(csv_path)
df.head()

,imageID,binaryLabels
0,image_0.jpg,0
1,image_1.jpg,0
2,image_3.jpg,0
3,image_4.jpg,0
4,image_5.jpg,0


In [4]:
X = []
labels = []
image_ids = []
image_paths = []
missing_images = []

In [5]:
for _, row in df.iterrows():
    image_id = str(row["imageID"]).strip()
    image_path = os.path.join(image_dir, image_id)

    if not os.path.isfile(image_path):
        missing_images.append(image_id)
        continue

    image = cv2.imread(image_path)

    if image is None:
        missing_images.append(image_id)
        continue

    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    X.append(image)
    labels.append(int(row["binaryLabels"]))
    image_ids.append(image_id)
    image_paths.append(image_path)

In [6]:
X = np.array(X, dtype=np.float32) / 255.0
labels = np.array(labels, dtype=np.int32)

In [7]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train, X_temp, labels_train, labels_temp = train_test_split(
    X,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Second split: 15% validation, 15% test
X_val, X_test, labels_val, labels_test = train_test_split(
    X_temp,
    labels_temp,
    test_size=0.50,
    random_state=42,
    stratify=labels_temp
)

print("Training images:", X_train.shape)
print("Validation images:", X_val.shape)
print("Test images:", X_test.shape)

Training images: (714, 224, 224, 3)
Validation images: (153, 224, 224, 3)
Test images: (153, 224, 224, 3)


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# MLflow setup
import mlflow
import mlflow.tensorflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Glaucoma_Classification")


mlflow_candidates = []

def log_model(model, run_name, y_true, y_pred, params=None):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("model_name", run_name)
        for key, value in (params or {}).items():
            mlflow.log_param(key, value)
        mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
        mlflow.log_metric("precision_macro", precision_score(y_true, y_pred, average="macro", zero_division=0))
        mlflow.log_metric("recall_macro", recall_score(y_true, y_pred, average="macro", zero_division=0))
        mlflow.log_metric("recall_normal", recall_score(y_true, y_pred, labels=[0], average="macro", zero_division=0))
        mlflow.log_metric("recall_glaucoma", recall_score(y_true, y_pred, labels=[1], average="macro", zero_division=0))
        mlflow.log_metric("f1_macro", f1_score(y_true, y_pred, average="macro", zero_division=0))
        mlflow.tensorflow.log_model(model, artifact_path="model")
        model_uri = f"runs:/{run.info.run_id}/model"
        run_id = run.info.run_id

    candidate = {
        "name": run_name,
        "model_uri": model_uri,
        "run_id": run_id,
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_normal": recall_score(y_true, y_pred, labels=[0], average="macro", zero_division=0),
        "recall_glaucoma": recall_score(y_true, y_pred, labels=[1], average="macro", zero_division=0),
    }
    mlflow_candidates.append(candidate)
    print(f"Saved run in MLflow: {run_name} | Macro Recall: {candidate['macro_recall']:.4f}")


C:\Users\dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),


    layers.Dense(1, activation="sigmoid")
])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,889 (429.25 KB)

 Trainable params: 109,889 (429.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [13]:
history = model.fit(
    X_train,
    labels_train,
    validation_data=(X_val, labels_val),
    epochs=10,
    batch_size=16,
    verbose=1
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 227ms/step - accuracy: 0.7003 - loss: 0.6249 - val_accuracy: 0.7059 - val_loss: 0.6368
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 230ms/step - accuracy: 0.7101 - loss: 0.6184 - val_accuracy: 0.7059 - val_loss: 0.6002
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 237ms/step - accuracy: 0.7101 - loss: 0.6118 - val_accuracy: 0.7059 - val_loss: 0.6008
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 242ms/step - accuracy: 0.7101 - loss: 0.6126 - val_accuracy: 0.7059 - val_loss: 0.6041
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 240ms/step - accuracy: 0.7101 - loss: 0.6121 - val_accuracy: 0.7059 - val_loss: 0.6019
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 247ms/step - accuracy: 0.7101 - loss: 0.6127 - val_accuracy: 0.7059 - val_loss: 0.6105
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 252ms/step - accuracy: 0.7101 - loss: 0.6068 - val_accuracy: 0.7059 - val_loss: 0.6045
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 12s 263ms/step - accuracy: 0.7101 - loss: 0.6080 - val_accu

In [15]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [16]:
val_probs = model.predict(X_val, verbose=0)
val_preds = (val_probs.ravel() >= 0.5).astype(int)
print("Confusion Matrix:")
print(confusion_matrix(labels_val, val_preds, labels=[0, 1]))

print("\nClassification Report:")
print(
    classification_report(
        labels_val,
        val_preds,
        labels=[0, 1],
        zero_division=0
    )
)

Confusion Matrix:
[[108   0]
 [ 45   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      1.00      0.83       108
           1       0.00      0.00      0.00        45

    accuracy                           0.71       153
   macro avg       0.35      0.50      0.41       153
weighted avg       0.50      0.71      0.58       153



In [18]:
log_model(
    model, "CNN_Baseline", labels_test, val_preds,
    params={"architecture": "CNN"}
)

2026/10/29 15:38:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 15:38:46 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Saved run in MLflow: CNN_Baseline | Macro Recall: 0.5000


mobile_net using focal loss

In [19]:
from tensorflow.keras import layers, Sequential
from tensorflow.keras.applications import MobileNetV2

# Load pretrained MobileNetV2
base_model_weighted = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base_model_weighted.trainable = False


In [20]:
from sklearn.utils.class_weight import compute_class_weight


classes = np.unique(labels_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=labels_train
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{np.int32(0): np.float64(0.7041420118343196), np.int32(1): np.float64(1.7246376811594204)}


In [22]:

from sklearn.utils.class_weight import compute_class_weight
# Get the class weights you already calculated
w0 = class_weights[0]
w1 = class_weights[1]

# Normalize alpha using the class weights
alpha = w1 / (w0 + w1)

print("Class 0 weight:", w0)
print("Class 1 weight:", w1)
print("Alpha for class 1:", alpha)

focal_loss = tf.keras.losses.BinaryFocalCrossentropy(
    apply_class_balancing=True,
    alpha=alpha,
    gamma=2.0,
    from_logits=False
)


Class 0 weight: 0.7041420118343196
Class 1 weight: 1.7246376811594204
Alpha for class 1: 0.7100840336134453


In [23]:
# Build a new model
mobile_net_weighted_focal_model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(scale=2.0, offset=-1.0),
    base_model_weighted,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
], name="Glaucoma_MobileNetV2_Focal_Weights")

In [24]:
mobile_net_weighted_focal_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=focal_loss,
    metrics=[
        "accuracy",
    ]
)

In [25]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [26]:
history_focal = mobile_net_weighted_focal_model.fit(
    X_train,
    labels_train,
    validation_data=(X_val, labels_val),
    epochs=10,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 15s 257ms/step - accuracy: 0.4594 - loss: 0.1467 - val_accuracy: 0.5425 - val_loss: 0.0820
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 217ms/step - accuracy: 0.4818 - loss: 0.1270 - val_accuracy: 0.5359 - val_loss: 0.0811
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 213ms/step - accuracy: 0.4650 - loss: 0.1400 - val_accuracy: 0.5490 - val_loss: 0.0805
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 216ms/step - accuracy: 0.5238 - loss: 0.1323 - val_accuracy: 0.4967 - val_loss: 0.0800
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 222ms/step - accuracy: 0.4902 - loss: 0.1271 - val_accuracy: 0.4967 - val_loss: 0.0797
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 225ms/step - accuracy: 0.4734 - loss: 0.1234 - val_accuracy: 0.5425 - val_loss: 0.0784
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 223ms/step - accuracy: 0.5224 - loss: 0.1167 - val_accuracy: 0.5163 - val_loss: 0.0778
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 213ms/step - accuracy: 0.5252 - loss: 0.1159 - val_accu

In [27]:
pred=mobile_net_weighted_focal_model.predict(X_test)
pred = (pred >= 0.6).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(labels_test, pred)
print("Accuracy:", accuracy)


5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 600ms/step
Accuracy: 0.6470588235294118


In [28]:

confusion_matrix(labels_test, pred)

array([[91, 18],
       [36,  8]])

In [29]:
from sklearn.metrics import classification_report
print(classification_report(labels_test, pred))

              precision    recall  f1-score   support

           0       0.72      0.83      0.77       109
           1       0.31      0.18      0.23        44

    accuracy                           0.65       153
   macro avg       0.51      0.51      0.50       153
weighted avg       0.60      0.65      0.62       153



In [30]:
log_model(
    mobile_net_weighted_focal_model, "MobileNetV2_Focal_ClassWeights",
    labels_test, pred,
    params={"architecture": "MobileNetV2", "loss": "focal_loss", "class_weights": True}
)

2026/10/29 15:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 15:41:19 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Saved run in MLflow: MobileNetV2_Focal_ClassWeights | Macro Recall: 0.5083


finetuning

In [32]:

# Freeze all layers first
base_model_weighted.trainable = True

# Unfreeze only the last 30 layers
for layer in base_model_weighted.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen
for layer in base_model_weighted.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile the model with a small learning rate
mobile_net_weighted_focal_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=focal_loss,
    metrics=[
        "accuracy",
       
    ]
)



In [33]:

history_finetune = mobile_net_weighted_focal_model.fit(
    X_train,
    labels_train,
    validation_data=(X_val, labels_val),
    epochs=15,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 16s 271ms/step - accuracy: 0.4888 - loss: 0.1054 - val_accuracy: 0.5490 - val_loss: 0.0724
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 253ms/step - accuracy: 0.5308 - loss: 0.0770 - val_accuracy: 0.6078 - val_loss: 0.0723
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 12s 256ms/step - accuracy: 0.5574 - loss: 0.0727 - val_accuracy: 0.5229 - val_loss: 0.0716
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 251ms/step - accuracy: 0.5896 - loss: 0.0698 - val_accuracy: 0.5686 - val_loss: 0.0720
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 251ms/step - accuracy: 0.6597 - loss: 0.0674 - val_accuracy: 0.5490 - val_loss: 0.0718
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 252ms/step - accuracy: 0.6232 - loss: 0.0669 - val_accuracy: 0.4967 - val_loss: 0.0717


In [34]:
out=mobile_net_weighted_focal_model.predict(X_test)
out = (out >= 0.5).astype(int)
accuracy = accuracy_score(labels_test, out)
print("Accuracy:", accuracy)
confusion_matrix(labels_test, out)

4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/stepWARNING:tensorflow:5 out of the last 16 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x0000029DF92C1800> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 605ms/step
Accuracy: 0.49673202614379086


array([[62, 47],
       [30, 14]])

In [35]:
print(classification_report(labels_test, out))

              precision    recall  f1-score   support

           0       0.67      0.57      0.62       109
           1       0.23      0.32      0.27        44

    accuracy                           0.50       153
   macro avg       0.45      0.44      0.44       153
weighted avg       0.55      0.50      0.52       153



In [36]:
log_model(
    mobile_net_weighted_focal_model, "MobileNetV2_FineTuning",
    labels_test, out,
    params={"architecture": "MobileNetV2", "fine_tuning": True}
)

2026/10/29 15:43:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 15:43:10 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Saved run in MLflow: MobileNetV2_FineTuning | Macro Recall: 0.4435


augmenatation +mobilenet

In [41]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    ),
    layers.RandomContrast(0.1)
], name="data_augmentation")

In [42]:
base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base.trainable = False

In [43]:
inputs = layers.Input(shape=(224, 224, 3))

# Apply augmentation only during training
x = data_augmentation(inputs)

# Convert pixels from [0, 255] to [-1, 1]
x = layers.Rescaling(
    scale=2.0, offset=-1.0
)(x)

# Extract features using MobileNetV2
x = base(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

model_aug = Model(inputs, outputs)

model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=focal_loss,
    metrics=[
        "accuracy",
    ]
)

In [44]:
model_aug.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [45]:
history_aug = model_aug.fit(
  X_train,
      labels_train,
      validation_data=(X_val, labels_val),
      epochs=15,
      batch_size=16,
      callbacks=[early_stopping],
      verbose=1
)

Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 15s 255ms/step - accuracy: 0.5112 - loss: 0.1390 - val_accuracy: 0.4314 - val_loss: 0.0865
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 230ms/step - accuracy: 0.5168 - loss: 0.1161 - val_accuracy: 0.6928 - val_loss: 0.0951
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 240ms/step - accuracy: 0.5056 - loss: 0.1334 - val_accuracy: 0.6601 - val_loss: 0.0868
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 241ms/step - accuracy: 0.5266 - loss: 0.1054 - val_accuracy: 0.6667 - val_loss: 0.0919


In [46]:
out_aug=model_aug.predict(X_test)
out_aug = (out_aug >= 0.5).astype(int)
accuracy = accuracy_score(labels_test, out_aug)
print("Accuracy:", accuracy)
confusion_matrix(labels_test, out_aug)

5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 572ms/step
Accuracy: 0.35947712418300654


array([[18, 91],
       [ 7, 37]])

In [47]:
print(classification_report(labels_test,out_aug))

              precision    recall  f1-score   support

           0       0.72      0.17      0.27       109
           1       0.29      0.84      0.43        44

    accuracy                           0.36       153
   macro avg       0.50      0.50      0.35       153
weighted avg       0.60      0.36      0.32       153



In [48]:
log_model(
    model_aug, "MobileNetV2_Augmentation",
    labels_test, out_aug,
    params={"architecture": "MobileNetV2", "augmentation": True}
)

2026/10/29 15:45:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 15:45:15 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Saved run in MLflow: MobileNetV2_Augmentation | Macro Recall: 0.5030


augmentation+finetuning(mobile_net)

In [50]:
base.trainable = True

# Freeze all layers except the last 30
for layer in base.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Trainable layers:", sum(
    layer.trainable for layer in base.layers
))

Trainable layers: 19


In [51]:
model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=focal_loss,
    metrics=[
        "accuracy",
    ]
)

In [52]:
history_aug_finetune = model_aug.fit(
  X_train,
      labels_train,
      validation_data=(X_val, labels_val),
      epochs=15,
      batch_size=16,
      callbacks=[early_stopping],
      verbose=1
)

Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 288ms/step - accuracy: 0.5126 - loss: 0.1146 - val_accuracy: 0.4706 - val_loss: 0.0788
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 12s 265ms/step - accuracy: 0.4846 - loss: 0.0873 - val_accuracy: 0.4183 - val_loss: 0.0768
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 12s 266ms/step - accuracy: 0.5098 - loss: 0.0784 - val_accuracy: 0.5229 - val_loss: 0.0753
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 255ms/step - accuracy: 0.5420 - loss: 0.0724 - val_accuracy: 0.5882 - val_loss: 0.0754
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 284ms/step - accuracy: 0.5910 - loss: 0.0709 - val_accuracy: 0.6078 - val_loss: 0.0752
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 279ms/step - accuracy: 0.5210 - loss: 0.0732 - val_accuracy: 0.5948 - val_loss: 0.0745
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 285ms/step - accuracy: 0.5378 - loss: 0.0717 - val_accuracy: 0.5229 - val_loss: 0.0736
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 279ms/step - accuracy: 0.5672 - loss: 0.0714 - val_accu

In [53]:
aug_out_FUN=model_aug.predict(X_test)
aug_out_FUN = (aug_out_FUN >= 0.5).astype(int)
accuracy = accuracy_score(labels_test, aug_out_FUN)
print("Accuracy:", accuracy)
confusion_matrix(labels_test, aug_out_FUN)

5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 582ms/step
Accuracy: 0.49019607843137253


array([[51, 58],
       [20, 24]])

In [54]:
print(classification_report(labels_test, aug_out_FUN))

              precision    recall  f1-score   support

           0       0.72      0.47      0.57       109
           1       0.29      0.55      0.38        44

    accuracy                           0.49       153
   macro avg       0.51      0.51      0.47       153
weighted avg       0.60      0.49      0.51       153



In [55]:
log_model(
    model_aug, "MobileNetV2_Augmentation_FineTuning",
    labels_test, aug_out_FUN,
    params={"architecture": "MobileNetV2", "augmentation": True, "fine_tuning": True}
)

2026/10/29 15:48:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 15:48:25 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Saved run in MLflow: MobileNetV2_Augmentation_FineTuning | Macro Recall: 0.5067


In [56]:

if not mlflow_candidates:
    raise RuntimeError("No models have been logged. Run the training and evaluation cells first.")

best = max(mlflow_candidates, key=lambda item: item["macro_recall"])
registered = mlflow.register_model(
    model_uri=best["model_uri"],
    name="Glaucoma_Classifier"
)

print("Registered only the best model.")
print("Model:", best["name"])
print("Registry name: Glaucoma_Classifier")
print("Registry version:", registered.version)
print("Macro Recall:", round(best["macro_recall"], 4))
print("Recall Normal:", round(best["recall_normal"], 4))
print("Recall Glaucoma:", round(best["recall_glaucoma"], 4))


Successfully registered model 'Glaucoma_Classifier'.
2026/10/29 15:49:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 1f6f66e3ebd140d4816d5bf25efc5aad has no artifacts at artifact path 'model', registering model based on models:/m-e2080336f2dc49769668b3d5df72751a instead


Registered only the best model.
Model: MobileNetV2_Focal_ClassWeights
Registry name: Glaucoma_Classifier
Registry version: 1
Macro Recall: 0.5083
Recall Normal: 0.8349
Recall Glaucoma: 0.1818


Created version '1' of model 'Glaucoma_Classifier'.
